# Setup

In [ ]:
import os
import numpy as np

from typing import List, Tuple
from pathlib import Path
from dotenv import load_dotenv


### Normalizing Paths

In [ ]:
# normalize paths for different OS
BASE_DIR = Path.cwd()
PDF_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"

### Locating our Data Sources

In [ ]:
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))

for path in pdf_paths:
    print(path)

### Importing some RAG Components

- `PyPDFLoader` (loads PDFs page-by-page and stores page metadata)
- `RecursiveCharacterTextSplitter` (chunks text)
- `OpenAIEmbeddings` (turns text into vectors)
- `Chroma` (stores vectors + metadata)
- `ChatOpenAI` (LLM for answering)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader            # loads PDFs page-by-page and stores page metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter     # Text splitter for documents
from langchain_openai import ChatOpenAI, OpenAIEmbeddings               # LLM for answering & Embedding tool to turn text into vectors
from langchain_chroma import Chroma                                     # Chroma vector store

# Step 1: Ingest & Parse

We load each PDF **page-by-page** so we can cite sources later:
- filename (`source`)
- page number (`page`)


In [ ]:
from langchain_core.documents import Document
import re

def clean_pdf_text(text: str) -> str:
    """Clean up common PDF extraction artifacts."""
    # Replace common PDF line break patterns
    text = re.sub(r'\n \n', ' ', text)  # "\n \n" -> " "
    # text = re.sub(r'\n \n', '\n\n', text)  # preserve paragraph boundary
    text = re.sub(r'\n\n+', '\n\n', text)  # Multiple newlines -> double newline
    text = re.sub(r' +', ' ', text)  # Multiple spaces -> single space
    text = text.strip()  # Remove leading/trailing whitespace
    return text

def load_pdfs(paths: List[Path]) -> List[Document]:
    all_docs: List[Document] = []
    for path in paths:
        loader = PyPDFLoader(str(path))
        docs = loader.load()  # typically one Document per page
        # Normalize metadata.source to the PDF filename for nicer printing
        for d in docs:
            d.metadata["source"] = Path(d.metadata.get("source", path)).name
            # Clean up the extracted text
            d.page_content = clean_pdf_text(d.page_content)
        all_docs.extend(docs)
    return all_docs

docs = load_pdfs(pdf_paths)
print(f"Loaded {len(docs)} page-documents.")

### Inspecting a loaded Document

In [ ]:
doc = docs[29]
print(f"Document type: {type(doc)}")
print(f"Document class: {doc.__class__}")

print("\n=== Public Attributes ===")
public_attrs = [attr for attr in dir(doc) if not attr.startswith('_')]
print(f"Public attributes: {public_attrs}")

In [ ]:
# main properties we care about for now
print(f"Has 'page_content': {hasattr(doc, 'page_content')}")
print(f"Has 'metadata': {hasattr(doc, 'metadata')}")

### Page Content - The Actual Information

In [ ]:
print("\n=== page_content ===")
print(f"Content type: {type(doc.page_content)}")
print(f"Content length: {len(doc.page_content)} characters")

print(f"\n=== page_content preview ===")
print(f"  {doc.page_content[:1000]}...")

### Metadata that helps us with structure

In [ ]:
print("\n=== metadata ===")
print(f"Metadata type: {type(doc.metadata)}")
print(f"Metadata keys: {list(doc.metadata.keys())}")

print("\n=== Sample ===")
print("Metadata sample:")
for key, value in doc.metadata.items():
    print(f"  {key}: {value}")

# Step 2: Chunking & Embedding

### We'll use OpenAI for this lab
Overview of available models: https://platform.openai.com/docs/models

In [ ]:
# read .env file and load environment variables
load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY not found. Create a .env file (copy from .env.example) and set OPENAI_API_KEY."
    )

# default models from .env file
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-4o-mini")
EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

# set models manually here
# CHAT_MODEL = ""
# EMBED_MODEL = ""

print("OPENAI_API_KEY found.")              # never print your API keys :)
print(f"Using chat model: {CHAT_MODEL}")
print(f"Using embed model: {EMBED_MODEL}")

### 2.1 Chunking
We'll use CharacterTextSplitter to show overlap, but RecursiveCharacterTextSplitter is usually better

In [ ]:
# CHUNK_SIZE = 1500
# CHUNK_OVERLAP = 150

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=CHUNK_SIZE,
#     chunk_overlap=CHUNK_OVERLAP,
# )

# chunks = splitter.split_documents(docs)



from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(chunk_size=3000, chunk_overlap=150, separator=" ")
chunks = splitter.split_documents(docs)


In [ ]:
print(f"Chunks created: {len(chunks)}")
print(f"Original pages: {len(docs)}")

In [ ]:
for key, value in chunks[1].metadata.items():
    print(f"  {key}: {value}")

print("\n" + chunks[1].page_content)

In [ ]:
for key, value in chunks[99].metadata.items():
    print(f"  {key}: {value}")

print("\n" + chunks[99].page_content)

In [ ]:
for key, value in chunks[100].metadata.items():
    print(f"  {key}: {value}")

print("\n" + chunks[100].page_content)

In [ ]:
ids = []
for i, d in enumerate(chunks):
    src = d.metadata.get("source", "unknown")
    page = d.metadata.get("page", "na")
    ids.append(f"{src}::p{page}::c{i}")

In [ ]:
for id in ids[:4]:
    print(id)

In [ ]:
# sizes = [(i, len(d.page_content), d.metadata.get("source"), d.metadata.get("page")) for i, d in enumerate(chunks)]
# sizes_sorted = sorted(sizes, key=lambda x: x[1], reverse=True)

# for i, n, src, page in sizes_sorted[:10]:
#     print(i, n, src, page)

# Step 3: Build Vector DB & Create Embeddings
We persist to `chroma_db/` so we can reload later without re-embedding.

In [ ]:
# create embeddings for all chunks
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    chunk_size=100,         # not to be confused with chunking size for text splitter!
    )

# assemble Chroma vector store from documents, embeddings, and ids
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,
    persist_directory=str(CHROMA_DIR),
)

try:
    vectorstore.persist()
except Exception:
    pass

print("Chroma vector DB is ready")
print(f"Stored chunks: {vectorstore._collection.count()}")

### Inspecting embeddings

In [ ]:
col = vectorstore._collection

data = col.get(
    include=["embeddings", "documents", "metadatas"],
    limit=3,
)

print(data.keys())
print("embedding length:", len(data["embeddings"][0]))
print("first 10 dims:", data["embeddings"][0][:10])

In [ ]:
print(len(data["embeddings"][0]))
print(data["embeddings"][0][:50])

# Step 4: Retrieval

### Similarity Comparison

In [ ]:
question = "Who is Frodo Baggins?"
question_embedding = embeddings.embed_query(question)

print("Question embedding length:", len(question_embedding))
print("First 10 dims:", question_embedding[:10])

In [ ]:
# lets grab 3 stored chunks
data = col.get(include=["embeddings", "documents"], limit=10)

chunk_embeddings = data["embeddings"]
chunk_docs = data["documents"]

# cosine similarity function
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity scores:\n")

for i, emb in enumerate(chunk_embeddings):
    score = cosine_similarity(question_embedding, emb)
    print(f"Chunk {i} similarity: {score:.4f}")
    print(f"Preview: {chunk_docs[i]}")
    print()

### Using the Chroma DB directly
In most Chroma setups, the score is Cosine distance (lower = more similar), not cosine similarity.
- Cosine similarity measures how aligned two vectors are.
- Cosine distance measures how far apart they are based on that alignment.

So for Cosine distance:
- 0.15  → very similar
- 0.80  → not similar

In [ ]:
results = vectorstore.similarity_search_with_score(question, k=3)

for i, (doc, score) in enumerate(results):
    print(f"Result {i}")
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print()

### Automated Retrieval

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_query = "Who is Frodo Baggins?"
retrieved = retriever.invoke(test_query)


In [ ]:
retrieved

In [ ]:
print(f"Query: {test_query}")
print(f"Retrieved chunks: {len(retrieved)}\n")

for i, doc in enumerate(retrieved, start=1):
    src = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "unknown")
    print(f"[{i}] {src} — page {page}")
    print(doc.page_content)
    print()

# Step 5: Querying


Key idea:
- The LLM is told to **only** use the retrieved context.
- If the context doesn’t contain the answer, it should say **“I don’t know.”**

In [ ]:
llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)

RAG_SYSTEM_PROMPT = '''You are a helpful assistant.
Answer the user's question using ONLY the provided context.
If the answer is not contained in the context, say: "I don't know."
Keep the answer concise and clear. Only answer questions about the provided context. Do not use any information that is not in the context.
'''

In [ ]:
def format_context(docs: List[Document], max_chars: int = 8000) -> str:
    """Concatenate retrieved chunks into one context string (truncate if needed)."""
    parts = []
    total = 0
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "unknown")
        header = f"\n\n---\nSOURCE: {src} | PAGE: {page}\n"
        text = d.page_content.strip()
        block = header + text
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "".join(parts).strip()

In [ ]:
def dedupe_sources(docs: List[Document]) -> List[Tuple[str, int]]:
    seen = set()
    out = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", None)
        key = (src, page)
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

In [ ]:
def rag_answer(question: str, k: int = 4) -> dict:
    retriever_k = vectorstore.as_retriever(search_kwargs={"k": k})
    retrieved_docs = retriever_k.invoke(question)

    if not retrieved_docs:
        return {"answer": "I don't know.", "sources": []}

    context = format_context(retrieved_docs)

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"},
    ]

    response = llm.invoke(messages)
    sources = dedupe_sources(retrieved_docs)

    return {"answer": response.content, "sources": sources, "retrieved_docs": retrieved_docs}


### Ask a Question

In [ ]:
question = "Does the story end happily for Boromir?"
result = rag_answer(question, k=5)

print("Question:", question)
print("\nAnswer:\n", result["answer"])

print("\nSources used:")
for src, page in result["sources"]:
    print(f"- {src} — page {page}")


# Demo Section

In [ ]:
while True:
    q = input("\nAsk a question (empty to stop): ").strip()
    if not q:
        print("Stopping.")
        break

    out = rag_answer(q, k=4)
    print("\nAnswer:\n", out["answer"])

    print("\nSources used:")
    for src, page in out["sources"]:
        print(f"- {src} — page {page}")

# Step 6: Agentic RAG

### Let's set up guardrails for the answers to protect Frodo on his journey

In [ ]:
RING_GUARD_PROMPT = """You are a strict content checker.
Decide whether the assistant's answer reveals sensitive information about the location of the ring.
Mark UNSAFE if it includes any information about Frodo Baggins (where he is or where he is going).
Do not mark UNSAFE for any other information, only mark UNSAFE if it includes specific information about Frodo Baggins (where he is, directions).
Otherwise mark SAFE.

Respond with EXACTLY one word: SAFE or UNSAFE.
"""



In [ ]:
def ring_location_is_safe(question: str, answer: str) -> bool:
    verdict = llm.invoke([
        {"role": "system", "content": RING_GUARD_PROMPT},
        {"role": "user", "content": f"Question: {question}\n\nAnswer: {answer}"}
    ]).content.strip().upper()
    return verdict.startswith("SAFE")

def rag_answer_with_guard(question: str, k: int = 4) -> dict:
    result = rag_answer(question, k=k)

    if not ring_location_is_safe(question, result["answer"]):
        result["answer"] = "Nice try, Sauron."
        result["sources"] = []  # avoid showing sources for blocked answers

    return result

In [ ]:
question = "Does Gandalf have a horse?"
# question = "Where is Frodo Baggins?"
result = rag_answer_with_guard(question, k=5)

print("Question:", question)
print("\nAnswer:", result["answer"])
print("\nSources:", result["sources"])

# Step 7: Ideas for you to continue building on your own!

- **Cleaner ingestion & parsing**: Swap raw PDF text for structure-aware parsing (headings, page numbers, tables) and store metadata (book, chapter, section) for smarter retrieval and citations. 
 
- **Smarter chunking**: Move from fixed-size chunks to semantic/structure chunks (by headings/paragraphs) with small overlap; add a quick eval loop to find the sweet spot for chunk size vs. answer quality.  
- **Better embeddings & normalization**: Try a stronger embedding model, normalize text (quotes/dashes/whitespace), and consider a second representation (e.g., title/heading embedding) so queries match both content and structure.  
- **Hybrid retrieval**: Combine dense vectors + keyword search (BM25) and re-rank top results; often the biggest quality bump with minimal code changes.  
- **Reranking / “best chunks” selection**: Add a lightweight reranker that scores the top 20 chunks and keeps the best 3–5 for the final prompt—less noise, fewer hallucinations.  
- **Query rewriting & multi-query**: Before retrieval, have an LLM rewrite the question into 2–4 focused sub-queries (synonyms, character names, related terms), retrieve for each, then merge/deduplicate.  
- **Answer groundedness checks**: Add a second LLM call to verify every claim is supported by retrieved context; if not, revise with citations or respond “I don’t know.”  
- **Agentic flow beyond guardrails**: Expand the loop: plan → retrieve → answer → critique → refine (max 1–2 iterations), with a strict stop condition for predictability and speed.  
- **Conversation memory (but safe)**: Store only non-sensitive user preferences + short conversation summaries, and always re-ground factual answers in retrieved documents.  
- **Evaluation harness**: Create a small test set (10–30 Q/A pairs), track metrics like answer correctness + citation relevance, and re-run automatically when tuning chunking/retrieval.  
- **Latency & cost tuning**: Cache embeddings and retrieval per query; use smaller/cheaper models for rewriting/guardrails and the strongest model for final answers.  
- **UX improvements**: Show clickable citations, highlight which chunks were used, and add “Ask a follow-up” suggestions to guide users.
